### Day 09 · 模块与高级特性> **前置**: Day01-08 全部> **关键问题**: 代码破千行后,单文件不堪重负 —— 怎么把代码**拆分到多个文件**、让重复的 “计时/日志/资源管理”逻辑自动复用? 本节是工具箱补完课:包、生成器、上下文管理器、装饰器

**import 模块(完整路径)**最基础的导入方式。把整个模块引入当前命名空间后,通过 **模块名.成员名** 的方式访问其中的函数、常量或类。优点是不会和当前文件的名字冲突,缺点是多打几个字。

In [ ]:
# 导入 Python 标准库中的 math 模块
import math

# 通过 模块名.成员名 的方式调用 sqrt
result = math.sqrt(16)
print(result)               # 4.0

# 也可以访问模块中的常量,例如圆周率
print(math.pi)              # 3.141592653589793


**from ... import(直接使用)**把模块中的**某个成员**直接引入当前命名空间后,调用时**不需要**再写模块前缀。注意如果当前文件已经有同名变量会被覆盖。

In [ ]:
# 从 math 模块中只引入 sqrt 这一个函数
from math import sqrt

# 调用时直接写函数名,不再需要 math. 前缀
print(sqrt(25))             # 5.0
print(sqrt(2))              # 1.4142135623730951

# 可以同时引入多个成员,逗号分隔
from math import sqrt, pi
print(pi)                   # 3.141592653589793


**import ... as(别名)**当模块名太长、或与现有名字冲突时,用 **as** 给成员起一个更短的别名。这是团队协作中常见的约定,例如 `import numpy as np`。

In [ ]:
# 把 math 模块中的 factorial 命名为别名 fac
from math import factorial as fac

# 调用时使用别名 fac,等价于 math.factorial(5)
print(fac(5))               # 120
print(fac(10))              # 3628800

# 也可以给整个模块起别名,用短名访问成员
import math as m
print(m.sqrt(9))            # 3.0


**自定义模块与包结构**把代码放进 `.py` 文件就是**模块**,同目录直接 `import` 即可使用。多个模块放进带 `__init__.py` 的文件夹就是**包**。`__all__` 列表控制 `from 模块 import *` 时哪些被导出。

In [ ]:
# === 自定义模块 ===
# 文件 utils.py 内容:
#   def greet(name):
#       return f"你好, {name}"

# 主程序中: import utils
# utils.greet("小明")

# === 包结构 ===
# my_pkg/
# +-- __init__.py           <必须存在
# +-- tools.py
# +-- helpers.py
# 使用: from my_pkg.tools import foo

# === __all__ 控制导出 ===
# 在模块中: __all__ = ["public_fn"]
# 星号导入时只有 public_fn 可见
print("模块=py文件,包=含__init__.py的文件夹")


**生成器 yield 基础**普通函数 `return` 一次就彻底结束,而 **yield** 可以**暂停并产出一个值**,等下次调用时从暂停处继续执行。这种“产出但不退场”的机制称为**惰性计算**。

In [ ]:
def squares(n):
    # 生成器函数:用 yield 逐个产出平方数
    for i in range(n):
        yield i * i          # 产出后暂停,下次从这里恢复

# 调用生成器函数不会立即执行,
# 而是返回一个生成器对象
gen = squares(5)
print(type(gen))            # <class 'generator'>

# 用 next() 逐个取值
print(next(gen))            # 0
print(next(gen))            # 1
print(next(gen))            # 4

# 也可以用 for 循环遍历
for x in squares(3):
    print("for:", x)


**生成器的特性:一次性与 yield from**生成器**只能遍历一次**,再次遍历为空。`yield from` 可把工作委托给另一个可迭代对象,等价于 for + yield 的简写。

In [ ]:
def squares(n):
    for i in range(n):
        yield i * i

# 一次性:第二次遍历为空
gen = squares(3)
print(list(gen))            # [0, 1, 4]
print(list(gen))            # []  第二次空了

# yield from 委托
def my_range(n):
    # 等价于: for i in range(n): yield i
    yield from range(n)

print(list(my_range(5)))    # [0, 1, 2, 3, 4]

# yield from 组合多个可迭代对象
def combined():
    yield from range(3)
    yield from range(5, 8)

print(list(combined()))


**生成器与列表的取舍**列表把全部数据存进内存,支持 `len()` 和多次遍历;生成器逐个产出,**省内存但只遍历一次,且没有长度**。数据量大时用生成器,需要反复读时用列表。

In [ ]:
import sys

# 列表占内存大,生成器几乎不占
list_data = [i * i for i in range(1000)]
gen_data = (i * i for i in range(1000))
print("列表大小:", sys.getsizeof(list_data))
print("生成器大小:", sys.getsizeof(gen_data))

# 列表支持 len(),生成器不支持
print("列表长度:", len(list_data))   # 1000
# print(len(gen_data))  <- TypeError

# 列表可索引,生成器不可
print(list_data[0])         # 0
# gen_data[0] <- TypeError


**斐波那契生成器**斐波那契数列无限长,不可能用列表全部存下来。生成器可以**按需产出**,理论上能无限生成下去。这是"惰性计算"最直接的价值体现。

In [ ]:
def fib(n):
    # 生成前 n 个斐波那契数
    a, b = 0, 1              # 前两个初始值
    for _ in range(n):
        yield a              # 产出当前值
        a, b = b, a + b      # 更新为下一对值

# 产出前 10 个斐波那契数
for x in fib(10):
    print(x, end=" ")
print()

# 也可以用 list() 一次性取出所有值
print(list(fib(8)))


**生成器表达式**生成器也有表达式写法,语法和列表提导很像,只是把方括号 `[]` 换成圆括号 `()`。在需要**节省内存**时特别有用,比如作为函数参数直接传递。

In [ ]:
# 列表提导:一次性算出所有值,存入内存
squares_list = [x * x for x in range(10)]
print(squares_list)         # [0, 1, 4, 9, ...]

# 生成器表达式:圆括号,惰性计算
squares_gen = (x * x for x in range(10))
print(squares_gen)          # <generator object>

# 生成器表达式可直接作为函数参数
# 不需要先创建列表再传入,节省内存
total = sum(x * x for x in range(10))
print(total)                # 285

# 配合 list() 可以一次性取出所有值
vals = (x*x for x in range(5))
print(list(vals))           # [0, 1, 4, 9, 16]


**上下文管理器 __enter__ / __exit__**`with` 语句需要对象实现两个方法:**__enter__**(进入时自动调用)和**__exit__**(退出时自动调用)。`with open` 自动关文件就是这个协议的典型应用。

In [ ]:
import time

class Timer:
    # 自定义上下文管理器:自动计时

    def __enter__(self):
        # 进入 with 块时调用,记录开始时间
        self.start = time.time()
        return self          # 返回值起给 as 后的变量

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 退出 with 块时调用,打印耗时
        # exc_type/exc_val/exc_tb 是异常信息
        elapsed = time.time() - self.start
        print(f"耗时 {elapsed:.4f} 秒")

with Timer():
    sum(range(1000000))
# 输出:耗时 x.xxxx 秒


**__exit__ 的异常处理参数**`__exit__` 必须接受三个参数:**exc_type**(异常类型)、**exc_val**(异常值)、**exc_tb**(异常追踪)。没有异常时三个值都是 **None**。如果想吞掉异常,可以返回 **True**。

In [ ]:
class SafeDivide:
    # 演示 __exit__ 中如何处理异常

    def __enter__(self):
        print("进入安全除法")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 没有异常时 exc_type 是 None
        if exc_type is None:
            print("正常结束")
            return
        # 有异常时打印,返回 True 表示吞掉异常
        print(f"捕获异常: {exc_type.__name__}")
        return True          # 吞掉异常

# 正常情况,无异常
with SafeDivide():
    print(10 / 2)           # 5.0

# 触发除零异常,被 __exit__ 捕获并吞掉
with SafeDivide():
    print(10 / 0)           # 不会崩溃

print("程序继续运行")


**contextlib.contextmanager 装饰器**除了手写 `__enter__`/`__exit__`,Python 提供了 **@contextmanager** 装饰器,用普通函数(包含一个 yield)快速定义上下文管理器。yield 之前是进入逻辑,yield 之后是退出逻辑。

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer():
    # yield 之前 = __enter__ 逻辑
    start = time.time()
    try:
        yield                # 把控制权交给 with 块
    finally:
        # yield 之后 = __exit__ 逻辑
        # finally 保证了即使抛异常也会执行
        elapsed = time.time() - start
        print(f"耗时 {elapsed:.4f} 秒")

# 使用方式和手写类的完全一样
with timer():
    sum(range(1000000))
# 输出:耗时 x.xxxx 秒


**装饰器基础 @timer**装饰器本质是**接收函数、返回新函数**的高阶函数,语法糖 `@decorator` 让写法简洁。核心思想:**在不修改原函数代码的前提下,额外添加功能**。

In [ ]:
import time
import functools

def timer(fn):
    # 装饰器:给函数加计时,不修改原函数
    @functools.wraps(fn)     # 保留原函数元信息
    def wrapper(*args, **kwargs):
        # wrapper 代替原函数被调用
        start = time.time()  # 记录开始时间
        result = fn(*args, **kwargs)  # 调用原函数
        elapsed = time.time() - start
        print(f"{fn.__name__} 耗时 {elapsed:.4f} 秒")
        return result        # 返回原函数的结果
    return wrapper           # 返回包装后的函数

# @timer 是语法糖,等价于 slow = timer(slow)
@timer
def slow():
    time.sleep(0.1)          # 模拟耗时操作

slow()                      # slow 耗时 0.1xxx 秒


**functools.wraps 的重要性**装饰器会**替换**原函数的元信息(函数名、文档字符串等),被装饰后 `__name__` 会变成 `wrapper`,给调试和文档带来麻烦。用 **@functools.wraps(fn)** 可复制元信息,这是**写装饰器的必须习惯**。

In [ ]:
import functools

def my_decorator(fn):
    @functools.wraps(fn)     # 关键:保留元信息
    def wrapper(*args, **kwargs):
        # 这是包装函数的文档字符串
        return fn(*args, **kwargs)
    return wrapper

@my_decorator
def greet(name):
    # 问候函数
    return f"你好, {name}"

# 没有 @functools.wraps 时,
# __name__ 会变成 "wrapper"
# 有了 @functools.wraps 后,元信息被保留
print(greet.__name__)       # greet(不是 wrapper)
print(greet.__doc__)        # None(因原函数没写 docstring)


**带参数的装饰器与类方法应用**当装饰器本身需要接收参数时,需要**三层嵌套**:外层接收参数,中层接收函数,内层是真正的包装逻辑。装饰器也可用在类方法上,`*args` 自动处理 self 参数。

In [ ]:
import functools

# === 带参数的装饰器 ===
def log(level="INFO"):
    # 外层:接收装饰器的参数 level
    def decorator(fn):
        # 中层:接收被装饰的函数 fn
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            # 内层:真正的包装逻辑
            print(f"[{level}] 调用 {fn.__name__}")
            return fn(*args, **kwargs)
        return wrapper
    return decorator

@log("DEBUG")               # 等价于 add = log("DEBUG")(add)
def add(a, b):
    return a + b

print(add(1, 2))            # [DEBUG] 调用 add -> 3

# === 装饰器用于类方法 ===
def call_log(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        # args[0] 是 self,即类的实例
        print(f"调用方法 {fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper

class Calc:
    @call_log
    def add(self, a, b):
        return a + b

c = Calc()
print(c.add(3, 5))          # 调用方法 add -> 8


**多个装饰器叠加**一个函数可以叠加**多个装饰器**,执行顺序是**从下往上**(靠近函数的装饰器先执行)。理解这一点对排查生产环境 Bug 很重要。

In [ ]:
def deco1(fn):
    # 第一个装饰器
    def wrapper(*args, **kwargs):
        print("deco1 前")
        result = fn(*args, **kwargs)
        print("deco1 后")
        return result
    return wrapper

def deco2(fn):
    # 第二个装饰器
    def wrapper(*args, **kwargs):
        print("deco2 前")
        result = fn(*args, **kwargs)
        print("deco2 后")
        return result
    return wrapper

# 装饰顺序:先应用 deco2,再应用 deco1
@deco1
@deco2
def hello():
    print("hello")

hello()
# 输出: deco1前 deco2前 hello deco2后 deco1后


**今日小结****模块与包**: `import 模块`(带前缀)、`from 模块 import 成员`(直接用)、`import ... as 别名`(改名)。模块=`.py` 文件,包=含 `__init__.py` 的文件夹。`__all__` 控制星号导出。**生成器**: `yield` 逐个产出值,惰性计算,省内存。`next()` 手动取值,`for` 循环自动遍历。生成器**只能遍历一次**,没有长度,不能索引。`yield from` 委托给子可迭代对象。生成器表达式用圆括号。**上下文管理器**:`__enter__`/`__exit__` 实现 `with` 语句。`__exit__` 的三个 `exc_` 参数处理异常,返回 True 可吞掉。`@contextmanager` 用 `try/finally` 快速定义。**装饰器**: 接收函数返回新函数的高阶函数。`@functools.wraps(fn)` 必须写,保留元信息。带参数时三层嵌套。多个装饰器从下往上叠加。**易错点**:- 漏写 `@functools.wraps`,原函数名变成 `wrapper`- 把生成器当列表用,`len()` 报错或第二轮为空- `__exit__` 方法签名漏写三个 `exc_` 参数- 包目录缺少 `__init__.py`,IDE 不认识

**更多练习**- 当堂练:course/lesson09/in_class/practice01-06.py- 课后作业:course/lesson09/homework/task01-03.py